# LangChain OpenAI Functions Output Parsers Reference

Developer-facing statements defined in `langchain_core.output_parsers.openai_functions`.

# `OutputFunctionsParser: BaseGenerationOutputParser[Any]`

Extracts an OpenAI-style function call from a chat generation.

## Fields

```python
args_only: bool = True # Whether to return only the function-call arguments
```

## Constructor

```python
OutputFunctionsParser(
    *,
    args_only: bool = True, # Whether to return only the function-call arguments
) -> None
```

## Methods

### `parse_result`

Extracts the function-call data from the first generation.

```python
@override
parse_result(
    self,
    result: list[Generation], # Candidate generations for one model input
    *,
    partial: bool = False, # Accepted for parser compatibility
) -> Any # Arguments string or complete function-call dictionary
```

The first result must be a `ChatGeneration`. Otherwise, the method raises `OutputParserException`.

The method deep-copies `message.additional_kwargs["function_call"]`. A missing `"function_call"` entry raises `OutputParserException`.

When `args_only=True`, it returns the raw value stored under `"arguments"` without decoding the JSON string. When `args_only=False`, it returns the complete copied function-call dictionary.

The `partial` value does not change parsing behavior.

In [ ]:
import json # Import JSON support for creating function-call arguments

from langchain_core.exceptions import OutputParserException # Import the real LangChain parser exception
from langchain_core.messages import AIMessage # Import a real LangChain AI message
from langchain_core.output_parsers.openai_functions import OutputFunctionsParser # Import the real function-call parser
from langchain_core.outputs import ChatGeneration, Generation # Import chat and text generation classes


arguments = { # Create function-call arguments
    "city": "Delhi", # Provide the requested city
    "unit": "celsius", # Provide the requested temperature unit
}

message = AIMessage( # Create an OpenAI-style function-call message
    content="", # Leave normal message content empty
    additional_kwargs={ # Add provider-specific function-call data
        "function_call": { # Define the function call
            "name": "get_weather", # Provide the function name
            "arguments": json.dumps(arguments), # Store arguments as a raw JSON string
        }
    },
)

generation = ChatGeneration(message=message) # Wrap the message in a chat generation

args_parser = OutputFunctionsParser(args_only=True) # Return only the arguments

raw_arguments = args_parser.parse_result([generation]) # Extract the raw arguments string
print("Raw arguments:", raw_arguments) # Display the undecoded JSON string
print("Raw arguments type:", type(raw_arguments).__name__) # Confirm that it is a string

invoked_arguments = args_parser.invoke(message) # Parse through the runnable interface
print("Invoke result:", invoked_arguments) # Display the runnable result

async_arguments = await args_parser.ainvoke(message) # Parse asynchronously in Jupyter
print("Async result:", async_arguments) # Display the asynchronous result

full_parser = OutputFunctionsParser(args_only=False) # Return the full function call

full_function_call = full_parser.parse_result([generation]) # Extract the complete dictionary
print("\nFull function call:", full_function_call) # Display the function-call data

full_function_call["name"] = "changed_name" # Modify the returned copied dictionary
original_name = message.additional_kwargs["function_call"]["name"] # Read the original value

print("Modified returned name:", full_function_call["name"]) # Display the changed copied value
print("Original message name:", original_name) # Confirm the original was not changed

try: # Start non-chat-generation error handling
    text_generation = Generation(text="normal text output") # Create a normal text generation
    args_parser.parse_result([text_generation]) # Try parsing an unsupported generation
except OutputParserException as error: # Catch the expected parser exception
    print("\nNon-chat generation error:", error) # Display the error

try: # Start missing-function-call error handling
    normal_message = AIMessage(content="No function call is present.") # Create a normal message
    normal_generation = ChatGeneration(message=normal_message) # Wrap it in a generation
    args_parser.parse_result([normal_generation]) # Try extracting a missing function call
except OutputParserException as error: # Catch the expected parser exception
    print("Missing function-call error:", error) # Display the error

# `JsonOutputFunctionsParser: BaseCumulativeTransformOutputParser[Any]`

Parses the arguments of an OpenAI-style function call as JSON.

In cumulative streaming mode, the parser can return partial JSON values. When the inherited `diff` option is enabled, emitted differences are JSON Patch operation lists produced by `jsonpatch.make_patch()`.

## Fields

```python
strict: bool = False # Value forwarded to JSON decoding strictness
args_only: bool = True # Whether to return only the decoded arguments
```

## Constructor

```python
JsonOutputFunctionsParser(
    *,
    strict: bool = False, # Value forwarded to JSON decoding strictness
    args_only: bool = True, # Whether to return only the decoded arguments
) -> None
```

## Methods

### `parse_result`

Parses one chat generation containing OpenAI function-call data.

```python
parse_result(
    self,
    result: list[Generation], # Generation list that must contain exactly one item
    *,
    partial: bool = False, # Whether to accept incomplete JSON arguments
) -> Any # Parsed arguments, full function-call data, or None
```

The method requires exactly one result and requires that result to be a `ChatGeneration`. Violations raise `OutputParserException`.

A missing `"function_call"` entry returns `None` when `partial=True`; otherwise, it raises `OutputParserException`.

When `partial=True`, the `"arguments"` string is decoded with `parse_partial_json()`:

- With `args_only=True`, the parsed arguments value is returned.
- With `args_only=False`, the complete function-call mapping is returned with `"arguments"` replaced by its parsed value.
- A `JSONDecodeError` returns `None`.

When `partial=False`, the `"arguments"` string is decoded with `json.loads(..., strict=self.strict)`:

- With `args_only=True`, the parsed arguments value is returned.
- With `args_only=False`, the complete function-call mapping is returned with `"arguments"` replaced by its parsed value.
- A `JSONDecodeError` or `TypeError` raises `OutputParserException`.

A missing `"arguments"` key returns `None`.

### `parse`

Unsupported text-only parsing hook.

```python
parse(
    self,
    text: str, # Text that is not used by this chat-generation parser
) -> Any
```

The implementation always raises `NotImplementedError`.


In [ ]:
import json # Import JSON support for encoding function arguments

from langchain_core.exceptions import OutputParserException # Import the real parser exception
from langchain_core.messages import AIMessage # Import a real LangChain AI message
from langchain_core.output_parsers.openai_functions import JsonOutputFunctionsParser # Import the real JSON function parser
from langchain_core.outputs import ChatGeneration # Import ChatGeneration for parse_result


parser = JsonOutputFunctionsParser( # Create the parser
    strict=False, # Use normal non-strict JSON decoding
    args_only=True, # Return only the decoded arguments
)

arguments = { # Create function-call arguments
    "city": "Delhi", # Provide the requested city
    "unit": "celsius", # Provide the temperature unit
}

message = AIMessage( # Create an OpenAI-style function-call message
    content="", # Leave normal message content empty
    additional_kwargs={ # Add provider-specific function-call data
        "function_call": { # Define the function call
            "name": "get_weather", # Provide the function name
            "arguments": json.dumps(arguments), # Encode arguments as JSON
        }
    },
)

generation = ChatGeneration(message=message) # Wrap the message in a chat generation

parsed_arguments = parser.parse_result([generation]) # Return only decoded arguments
print("Arguments only:", parsed_arguments) # Display the decoded arguments

invoked_result = parser.invoke(message) # Parse through the runnable interface
print("Invoke result:", invoked_result) # Display the runnable result

async_result = await parser.ainvoke(message) # Parse asynchronously in Jupyter
print("Async result:", async_result) # Display the asynchronous result

full_parser = JsonOutputFunctionsParser( # Create another parser
    strict=False, # Use normal JSON decoding
    args_only=False, # Return the complete function-call dictionary
)

full_result = full_parser.parse_result([generation]) # Parse the complete function call
print("\nComplete function call:", full_result) # Display the full result

partial_message = AIMessage( # Create a message with incomplete JSON
    content="", # Leave normal message content empty
    additional_kwargs={ # Add function-call data
        "function_call": { # Define the partial function call
            "name": "get_weather", # Provide the function name
            "arguments": '{"city": "Delhi", "unit": "cel', # Provide incomplete JSON
        }
    },
)

partial_generation = ChatGeneration(message=partial_message) # Wrap the partial message

partial_result = parser.parse_result( # Parse incomplete arguments
    [partial_generation], # Provide exactly one generation
    partial=True, # Enable partial JSON parsing
)

print("Partial result:", partial_result) # Display the recovered partial data

missing_call_message = AIMessage(content="No function call is present.") # Create a normal message
missing_call_generation = ChatGeneration(message=missing_call_message) # Wrap the message

missing_partial_result = parser.parse_result( # Parse a missing function call
    [missing_call_generation], # Provide the generation
    partial=True, # Treat it as incomplete output
)

print("Missing function call in partial mode:", missing_partial_result) # Display None

invalid_message = AIMessage( # Create a message with malformed JSON
    content="", # Leave normal message content empty
    additional_kwargs={ # Add function-call data
        "function_call": { # Define the invalid function call
            "name": "get_weather", # Provide the function name
            "arguments": '{"city": Delhi}', # Provide malformed JSON
        }
    },
)

invalid_generation = ChatGeneration(message=invalid_message) # Wrap the invalid message

try: # Start complete-result error handling
    parser.parse_result([invalid_generation]) # Parse malformed complete JSON
except OutputParserException as error: # Catch the LangChain parser exception
    print("\nComplete JSON error:", error) # Display the error

try: # Start text-only method handling
    parser.parse('{"city": "Delhi"}') # Call the unsupported parse method
except NotImplementedError: # Catch the expected error
    print("parse() is not supported by this parser.") # Explain the result

# `JsonKeyOutputFunctionsParser: JsonOutputFunctionsParser`

Parses JSON function-call arguments and returns one selected key.

## Fields

```python
key_name: str # Name of the JSON key to return
```

## Constructor

```python
JsonKeyOutputFunctionsParser(
    *,
    strict: bool = False, # Value forwarded to JSON decoding strictness
    args_only: bool = True, # Whether the parent parser returns only arguments
    key_name: str, # Name of the JSON key to return
) -> None
```

## Methods

### `parse_result`

Returns one key from the value parsed by `JsonOutputFunctionsParser`.

```python
parse_result(
    self,
    result: list[Generation], # Generation list passed to the parent parser
    *,
    partial: bool = False, # Whether to perform partial parsing
) -> Any # Selected value or None
```

When partial parsing produces `None`, the method returns `None`.

For partial results, it uses `res.get(key_name)`, so an unavailable key returns `None`. For complete results, it uses `res[key_name]`, so an unavailable key raises `KeyError`.

In [ ]:
import json # Import JSON support for encoding function arguments

from langchain_core.messages import AIMessage # Import a real LangChain AI message
from langchain_core.output_parsers.openai_functions import JsonKeyOutputFunctionsParser # Import the real key parser
from langchain_core.outputs import ChatGeneration # Import ChatGeneration for parse_result


parser = JsonKeyOutputFunctionsParser( # Create the JSON key parser
    key_name="city", # Return only the city value
    strict=False, # Allow normal non-strict JSON decoding
    args_only=True, # Parse only the function-call arguments
)

function_arguments = { # Create function-call arguments
    "name": "Saad", # Provide the person's name
    "age": 22, # Provide the person's age
    "city": "Delhi", # Provide the city
}

message = AIMessage( # Create an OpenAI-style function-call message
    content="", # Leave normal message content empty
    additional_kwargs={ # Add provider-specific function-call data
        "function_call": { # Define the function call
            "name": "extract_person", # Provide the function name
            "arguments": json.dumps(function_arguments), # Encode arguments as JSON
        }
    },
)

generation = ChatGeneration(message=message) # Wrap the message in a chat generation

city = parser.parse_result([generation]) # Parse the generation and return only city
print("City from parse_result():", city) # Display the selected value

city_from_invoke = parser.invoke(message) # Parse through the runnable interface
print("City from invoke():", city_from_invoke) # Display the selected value

async_city = await parser.ainvoke(message) # Parse asynchronously in Jupyter
print("City from ainvoke():", async_city) # Display the selected value

partial_parser = JsonKeyOutputFunctionsParser( # Create a parser for a missing key
    key_name="country", # Request a key not available in the JSON
)

partial_result = partial_parser.parse_result( # Parse using partial-result behavior
    [generation], # Provide the generation
    partial=True, # Allow partial parsing
)

print("Missing key in partial result:", partial_result) # Display None

try: # Start complete-result error handling
    partial_parser.parse_result([generation]) # Parse with the missing key
except KeyError as error: # Catch the missing-key error
    print("Missing key in complete result:", error) # Display the KeyError

# `PydanticOutputFunctionsParser: OutputFunctionsParser`

Parses OpenAI function-call arguments into a Pydantic model.

A single Pydantic model class can be supplied, or a mapping from function names to model classes can select different schemas for different calls. Both Pydantic v2 and Pydantic v1 model classes are supported.

## Fields

```python
pydantic_schema: TypeBaseModel | dict[str, TypeBaseModel] # Schema or function-name-to-schema mapping
```

The inherited `args_only` field controls whether the base parser returns only the raw argument string or the complete function-call dictionary.

## Constructor

```python
PydanticOutputFunctionsParser(
    *,
    args_only: bool = True, # Whether to parse only the raw arguments string
    pydantic_schema: TypeBaseModel | dict[str, TypeBaseModel], # Schema or schema mapping
) -> None
```

When `args_only` is omitted, the pre-validation hook sets it to `True` only when `pydantic_schema` is a single Pydantic v2 `BaseModel` subclass. A schema dictionary therefore defaults to `args_only=False`.

## Methods

### `validate_schema`

Validates and adjusts the parser configuration before model construction.

```python
@model_validator(mode="before")
@classmethod
validate_schema(
    cls,
    values: dict[str, Any], # Values supplied to the parser constructor
) -> Any # Validated and possibly updated values
```

When `"args_only"` is absent, the method derives it from whether the supplied schema is a single Pydantic v2 model class.

When multiple schemas are supplied in a dictionary, explicitly setting `args_only=True` raises `ValueError`.

### `parse_result`

Parses function-call arguments into the selected Pydantic model.

```python
@override
parse_result(
    self,
    result: list[Generation], # Chat generations containing function-call data
    *,
    partial: bool = False, # Accepted but not forwarded to the parent parser
) -> Any # Validated Pydantic model instance
```

With `args_only=True`, the inherited parser returns the raw arguments string. The method validates that JSON using the single supplied schema.

With `args_only=False`, the inherited parser returns the complete function-call dictionary. For a schema mapping, the function-call `"name"` selects the model class; otherwise, the single supplied schema is used.

Pydantic v2 schemas are parsed with `model_validate_json()`. Pydantic v1 schemas are parsed with `parse_raw()`.

Unsupported schema types and a dictionary schema combined with `args_only=True` raise `ValueError`. Pydantic validation errors propagate from the selected model.

The `partial` value is not forwarded to the parent implementation and does not enable partial Pydantic parsing.

In [ ]:
import json # Import JSON support for encoding function arguments

from pydantic import BaseModel # Import Pydantic's base model

from langchain_core.messages import AIMessage # Import a real LangChain AI message
from langchain_core.output_parsers.openai_functions import PydanticOutputFunctionsParser # Import the real parser
from langchain_core.outputs import ChatGeneration # Import ChatGeneration for parse_result


class WeatherRequest(BaseModel): # Define the schema for a weather function
    city: str # Store the requested city
    unit: str # Store the temperature unit


class CalculatorRequest(BaseModel): # Define the schema for a calculator function
    first_number: float # Store the first number
    second_number: float # Store the second number
    operation: str # Store the requested operation


single_parser = PydanticOutputFunctionsParser( # Create a single-schema parser
    pydantic_schema=WeatherRequest, # Validate arguments as WeatherRequest
    args_only=True, # Parse only the arguments JSON
)

weather_message = AIMessage( # Create an AI function-call message
    content="", # Leave the normal content empty
    additional_kwargs={ # Add function-call information
        "function_call": { # Define the function call
            "name": "get_weather", # Provide the function name
            "arguments": json.dumps({ # Encode arguments as JSON
                "city": "Delhi", # Provide the city
                "unit": "celsius", # Provide the unit
            }),
        },
    },
)

weather_generation = ChatGeneration(message=weather_message) # Wrap the message in a generation

weather_request = single_parser.parse_result([weather_generation]) # Parse into WeatherRequest
print("Single-schema result:", weather_request) # Display the validated model
print("Requested city:", weather_request.city) # Access a model attribute

invoke_result = single_parser.invoke(weather_message) # Parse through the runnable interface
print("Invoke result:", invoke_result) # Display the runnable result

async_result = await single_parser.ainvoke(weather_message) # Parse asynchronously in Jupyter
print("Async result:", async_result) # Display the asynchronous result

schema_mapping = { # Map function names to schemas
    "get_weather": WeatherRequest, # Select WeatherRequest for get_weather
    "calculate": CalculatorRequest, # Select CalculatorRequest for calculate
}

mapping_parser = PydanticOutputFunctionsParser( # Create a multi-schema parser
    pydantic_schema=schema_mapping, # Supply the schema mapping
) # args_only automatically becomes False

calculator_message = AIMessage( # Create a calculator function-call message
    content="", # Leave the normal content empty
    additional_kwargs={ # Add function-call information
        "function_call": { # Define the function call
            "name": "calculate", # Select CalculatorRequest
            "arguments": json.dumps({ # Encode arguments as JSON
                "first_number": 10, # Provide the first number
                "second_number": 5, # Provide the second number
                "operation": "multiply", # Provide the operation
            }),
        },
    },
)

calculator_generation = ChatGeneration(message=calculator_message) # Wrap the message in a generation

calculator_request = mapping_parser.parse_result([calculator_generation]) # Select and validate the model
print("\nMapping result:", calculator_request) # Display the validated model
print("Selected model:", type(calculator_request).__name__) # Display the selected schema
print("Operation:", calculator_request.operation) # Access a validated field

# `PydanticAttrOutputFunctionsParser: PydanticOutputFunctionsParser`

Parses function-call arguments into a Pydantic model and returns one attribute from that model.

## Fields

```python
attr_name: str # Name of the model attribute to return
```

## Constructor

```python
PydanticAttrOutputFunctionsParser(
    *,
    args_only: bool = True, # Whether to parse only the raw arguments string
    pydantic_schema: TypeBaseModel | dict[str, TypeBaseModel], # Schema or schema mapping
    attr_name: str, # Name of the model attribute to return
) -> None
```

## Methods

### `parse_result`

Returns one attribute from the Pydantic model produced by the parent parser.

```python
@override
parse_result(
    self,
    result: list[Generation], # Chat generations containing function-call data
    *,
    partial: bool = False, # Accepted but not forwarded to the parent parser
) -> Any # Selected model attribute
```

The method calls the parent parser and returns `getattr(parsed_model, attr_name)`. A missing attribute raises `AttributeError`.

The `partial` value is not forwarded to the parent implementation.

In [ ]:
import json # Import JSON support for encoding function arguments

from pydantic import BaseModel # Import BaseModel for defining the function schema

from langchain_core.messages import AIMessage # Import a real LangChain AI message
from langchain_core.output_parsers.openai_functions import PydanticAttrOutputFunctionsParser # Import the real attribute parser
from langchain_core.outputs import ChatGeneration # Import ChatGeneration for parse_result


class Person(BaseModel): # Define the Pydantic schema for function arguments
    name: str # Store the person's name
    age: int # Store the person's age
    city: str # Store the person's city


parser = PydanticAttrOutputFunctionsParser( # Create the parser
    pydantic_schema=Person, # Validate arguments using the Person model
    attr_name="age", # Return only the age attribute
    args_only=True, # Parse only the raw function arguments
)

function_arguments = { # Create function-call arguments
    "name": "Saad", # Provide the person's name
    "age": 22, # Provide the person's age
    "city": "Delhi", # Provide the person's city
}

message = AIMessage( # Create an AI message containing a function call
    content="", # Function-call messages usually contain no normal text
    additional_kwargs={ # Add provider-specific function-call data
        "function_call": { # Define the OpenAI-style function call
            "name": "extract_person", # Provide the function name
            "arguments": json.dumps(function_arguments), # Encode arguments as JSON
        }
    },
)

generation = ChatGeneration(message=message) # Wrap the message in a chat generation

age_from_result = parser.parse_result([generation]) # Parse and return only age
print("Age from parse_result():", age_from_result) # Display the extracted age

age_from_invoke = parser.invoke(message) # Parse through the runnable interface
print("Age from invoke():", age_from_invoke) # Display the extracted age

async_age = await parser.ainvoke(message) # Parse asynchronously in Jupyter
print("Age from ainvoke():", async_age) # Display the extracted age

city_parser = PydanticAttrOutputFunctionsParser( # Create another attribute parser
    pydantic_schema=Person, # Reuse the Person schema
    attr_name="city", # Return only the city attribute
    args_only=True, # Parse only the raw arguments
)

city = city_parser.parse_result([generation]) # Parse and extract the city
print("City:", city) # Display the extracted city